# Panzer — Endpoints autenticados (trading)

Este notebook muestra como usar `BinanceClient` para operar con endpoints
que requieren firma HMAC-SHA256: consultar cuenta, gestionar ordenes, etc.

**Requisitos:**
- API key y API secret de Binance (se almacenan cifrados en `~/.panzer_creds`).
- La primera vez que ejecutes, Panzer te pedira las credenciales por terminal.

> **ATENCION:** Los ejemplos de ordenes estan comentados para evitar operaciones
> accidentales. Descomentarlos bajo tu propia responsabilidad.

## 1. Crear un cliente autenticado

`BinanceClient` extiende `BinancePublicClient` — tienes todos los endpoints
publicos mas los autenticados.

In [ ]:
from panzer import BinanceClient

# Primera vez: te pide api_key y api_secret por terminal.
# Siguientes veces: los lee de ~/.panzer_creds automaticamente.
client = BinanceClient(market="spot")

# Sincronizar reloj (importante para la firma)
client.ensure_time_offset_ready(min_samples=3)

print(f"Cliente autenticado listo (market={client.market})")

## 2. Consultar cuenta

Balances, permisos, tipo de cuenta, etc.

In [ ]:
account = client.account()

# Permisos de la API key
print(f"Puede operar:  {account.get('canTrade')}")
print(f"Puede retirar: {account.get('canWithdraw')}")
print(f"Puede depositar: {account.get('canDeposit')}")

# Balances con saldo > 0
balances = [b for b in account["balances"] if float(b["free"]) > 0 or float(b["locked"]) > 0]
print(f"\nBalances con saldo ({len(balances)}):")
for b in balances[:10]:
    print(f"  {b['asset']:>6s}  free={b['free']:>14s}  locked={b['locked']:>14s}")

## 3. Consultar trades propios

In [ ]:
my_trades = client.my_trades("BTCUSDT", limit=5)

if my_trades:
    print(f"Ultimos {len(my_trades)} trades propios en BTCUSDT:\n")
    for t in my_trades:
        side = "BUY " if t["isBuyer"] else "SELL"
        print(f"  {side}  price={t['price']:>12s}  qty={t['qty']:>12s}  commission={t['commission']}")
else:
    print("No hay trades propios en BTCUSDT.")

## 4. Ordenes abiertas

In [ ]:
# Ordenes abiertas para un simbolo
open_orders = client.open_orders("BTCUSDT")

if open_orders:
    print(f"{len(open_orders)} ordenes abiertas en BTCUSDT:\n")
    for o in open_orders:
        print(f"  orderId={o['orderId']}  {o['side']}  {o['type']}  price={o['price']}  qty={o['origQty']}")
else:
    print("No hay ordenes abiertas en BTCUSDT.")

## 5. Historial de ordenes

In [ ]:
# Todas las ordenes (abiertas, cerradas, canceladas)
all_orders = client.all_orders("BTCUSDT", limit=5)

if all_orders:
    print(f"Ultimas {len(all_orders)} ordenes en BTCUSDT:\n")
    for o in all_orders:
        print(f"  {o['orderId']}  {o['side']:>4s}  {o['type']:<16s}  status={o['status']:<12s}  price={o['price']}")
else:
    print("No hay historial de ordenes en BTCUSDT.")

## 6. Crear y cancelar ordenes

> **CUIDADO:** Las siguientes celdas crean ordenes REALES.
> Estan comentadas por seguridad. Descomenta solo si sabes lo que haces.

### Orden LIMIT

In [ ]:
# DESCOMENTA PARA EJECUTAR (orden real):

# order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="LIMIT",
#     quantity=0.001,
#     price=50000.0,
#     time_in_force="GTC",
# )
# print(f"Orden creada: orderId={order['orderId']}  status={order['status']}")

### Orden MARKET

In [ ]:
# DESCOMENTA PARA EJECUTAR (orden real):

# market_order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="MARKET",
#     quote_order_qty=10.0,   # comprar 10 USDT de BTC
# )
# print(f"Orden market: orderId={market_order['orderId']}  fills={market_order.get('fills')}")

### Cancelar una orden

In [ ]:
# DESCOMENTA PARA EJECUTAR (cancelacion real):

# cancelled = client.cancel_order("BTCUSDT", order_id=12345678)
# print(f"Orden cancelada: {cancelled['orderId']}  status={cancelled['status']}")

## 7. Peticion generica firmada

Para endpoints autenticados sin wrapper, usa `signed_request()` directamente.

In [ ]:
# Ejemplo: obtener estado de un listen key (USER_STREAM, solo necesita API key)
# signed_request con sign=False solo inyecta el header X-MBX-APIKEY

# listen_key = client.signed_request(
#     "POST",
#     "/api/v3/userDataStream",
#     sign=False,  # USER_STREAM no necesita firma HMAC
# )
# print(f"Listen key: {listen_key['listenKey']}")

## 8. Futuros (USDT-M y COIN-M)

Exactamente la misma interfaz, solo cambia `market`.

In [ ]:
# Futuros USDT-M
# um_client = BinanceClient(market="um")
# um_account = um_client.account()
# print(f"UM Balance total: {um_account.get('totalWalletBalance')} USDT")

# Futuros COIN-M
# cm_client = BinanceClient(market="cm")
# cm_account = cm_client.account()
# print(f"CM assets: {len(cm_account.get('assets', []))}")

## 9. Manejo de errores en endpoints autenticados

Los errores de la API (credenciales invalidas, permisos insuficientes,
simbolo incorrecto, etc.) se capturan con `BinanceAPIException`.

In [ ]:
from panzer.errors import BinanceAPIException

try:
    # Intentar cancelar una orden que no existe
    client.cancel_order("BTCUSDT", order_id=999999999)
except BinanceAPIException as e:
    print(f"Error esperado:")
    print(f"  HTTP:    {e.status_code}")
    print(f"  Codigo:  {e.error_payload.code}")
    print(f"  Mensaje: {e.error_payload.msg}")

## Resumen de endpoints autenticados

| Metodo | HTTP | Descripcion |
|--------|------|-------------|
| `client.account()` | GET | Info de cuenta (balances, permisos) |
| `client.my_trades(symbol)` | GET | Historial de trades propios |
| `client.new_order(symbol, side, order_type, ...)` | POST | Crear orden |
| `client.cancel_order(symbol, order_id=)` | DELETE | Cancelar orden |
| `client.open_orders(symbol=)` | GET | Ordenes abiertas |
| `client.all_orders(symbol)` | GET | Historial completo de ordenes |
| `client.signed_request(method, endpoint, params)` | * | Cualquier endpoint firmado |

Todos los metodos comparten `recv_window` (ventana de validez en ms) y `timeout` (timeout HTTP).